# Data Preparation for the Bug Triage Agent

This notebook prepares the data used by `agentic_system.ipynb`. It is run once, and the agent notebook only loads the results.

**Source:** Eclipse bug reports, `eclipse_bug_report_data.csv` (AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset [https://huggingface.co/datasets/AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset]). 9,776 rows, 19 columns. Every bug has the resolution "fixed".

**Outputs**
- `data/index.csv`: 500 past bugs. The agent's duplicate search runs over these.
- `data/test.csv`: 21 test reports used to evaluate the agent.

**Steps:** load → keep columns → simplify labels → split → add hand-written test reports → check → save.

In [1]:
import pandas as pd
import re

from pathlib import Path

## 1. Load the data

In [2]:
df = pd.read_csv("eclipse_bug_report_data.csv")

## 2. Keep only the columns the agent needs

Kept: `bug_id`, `product_name`, `short_description` (title), `long_description`, `component_name`, `severity_category`.

`assignee_name` and `reporter_name` are dropped on purpose, so no personal names are sent to the LLM. The other columns (dates, votes, fix time, codes) are not needed.

149 missing descriptions are replaced with an empty string.

In [3]:
KEEP_COLS = [
    "bug_id",
    "product_name",
    "short_description",
    "long_description",
    "component_name",
    "severity_category",
]

def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    before = df.shape[1]
    df = df[KEEP_COLS]
    print(f"Columns: {before} -> {df.shape[1]}")
    return df

def fill_missing_description(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    missing = df["long_description"].isna().sum()
    df["long_description"] = df["long_description"].fillna("")
    print(f"Filled {missing} missing long_description values")
    return df

bugs = select_columns(df)
bugs = fill_missing_description(bugs)
bugs.head()

Columns: 19 -> 6
Filled 149 missing long_description values


,bug_id,product_name,short_description,long_description,component_name,severity_category
0,RECOMMENDERS-467951,RECOMMENDERS,LogTraceException in ProposalUtils.toMethodNam...,The following incident was reported via the au...,Core,normal
1,QVTO-463396,QVTO,CCE in DecorationNodeImpl.eSet (159),The following incident was reported via the au...,Engine,normal
2,EQUINOX-530069,EQUINOX,[http servlet] During dispatching javax.servle...,Original issue https://issues.liferay.com/brow...,Compendium,normal
3,NATTABLE-422482,NATTABLE,Left border of NatTable is not drawn,Rendering a NatTable on a Composite with margi...,Core,normal
4,WTP_JAVA_EE_TOOLS-116294,WTP_JAVA_EE_TOOLS,NPE while importing EAR with utility jar,Import and EAR which has EJb and Ejb client tr...,jst.j2ee,blocker


## 3. Simplify the labels

**Severity: 6 levels become 3.** `trivial` and `minor` become `low`. `major`, `critical` and `blocker` become `high`. `normal` stays. The data is very uneven: 79% of all bugs are `normal`, so an agent that always says "normal" would be right 79% of the time. This is the baseline the agent must beat.

**Components: top 5 only** (UI, Core, SWT, TPTP, Releng). This gives the triage agent a small, clear set to choose from. The agent also has an `other` option for everything else.

Result: 2,559 bugs (77% `normal`).

In [4]:
SEVERITY_MAP = {
    "trivial": "low",
    "minor": "low",
    "normal": "normal",
    "major": "high",
    "critical": "high",
    "blocker": "high",
}

def map_severity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["severity"] = df["severity_category"].map(SEVERITY_MAP)
    unmapped = df["severity"].isna().sum()
    print(f"Unmapped severity values: {unmapped}")
    return df.drop(columns=["severity_category"])

def filter_top_components(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    df = df.copy()
    top = df["component_name"].value_counts().head(n).index.tolist()
    before = len(df)
    df = df[df["component_name"].isin(top)]
    print(f"Top {n} components: {top}")
    print(f"Rows: {before} -> {len(df)} (dropped {before - len(df)})")
    return df

bugs = map_severity(bugs)
bugs = filter_top_components(bugs, n=5)
print(bugs["severity"].value_counts())
print(bugs["component_name"].value_counts())

Unmapped severity values: 0
Top 5 components: ['UI', 'Core', 'SWT', 'TPTP', 'Releng']
Rows: 9776 -> 2559 (dropped 7217)
severity
normal    1982
high       444
low        133
Name: count, dtype: int64
component_name
UI        956
Core      757
SWT       344
TPTP      307
Releng    195
Name: count, dtype: int64


## 4. Split into search index and test reports

The test reports are picked first (5 low, 5 normal, 5 high) and removed from the pool. The 500-bug index is then sampled from what remains. This way a test report can never appear in the agent's own search results.

The seed is fixed (`42`), so the split is repeatable. The checks at the end of the cell confirm that no bug ID or title appears in both sets.

In [5]:
SEED = 42

def split_index_and_test(
    df: pd.DataFrame,
    n_index: int = 500,
    n_per_severity: int = 5,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    test = df.groupby("severity", group_keys=False).sample(
        n=n_per_severity, random_state=seed
    )
    remaining = df.drop(test.index)
    index = remaining.sample(n=n_index, random_state=seed)
    print(f"Test: {len(test)} | Index: {len(index)}")
    print("Test severity mix:\n", test["severity"].value_counts())
    return index.reset_index(drop=True), test.reset_index(drop=True)

index_df, test_df = split_index_and_test(bugs)

# Safety checks
overlap_ids = set(index_df["bug_id"]) & set(test_df["bug_id"])
overlap_titles = set(index_df["short_description"]) & set(test_df["short_description"])
print(f"Overlapping bug_ids: {len(overlap_ids)}")
print(f"Overlapping titles: {len(overlap_titles)}")

Test: 15 | Index: 500
Test severity mix:
 severity
high      5
low       5
normal    5
Name: count, dtype: int64
Overlapping bug_ids: 0
Overlapping titles: 0


## 5. Find bugs to reword as duplicate tests

This cell only lists candidates from the index (clear title, real description, not an auto-generated report). I picked 3 by hand. It is exploratory, and its output is not used later.

In [6]:
def duplicate_candidates(index: pd.DataFrame, n: int = 10, seed: int = SEED) -> pd.DataFrame:
    df = index.copy()
    clean = df[
        (df["short_description"].str.len() > 30)
        & ~df["long_description"].str.contains("reported via the automated", case=False)
        & (df["long_description"].str.len() > 80)
    ]
    print(f"{len(clean)} usable candidates")
    return clean.sample(n=n, random_state=seed)

for _, row in duplicate_candidates(index_df).iterrows():
    print(row["bug_id"], "|", row["product_name"], "|", row["component_name"], "|", row["severity"])
    print("  TITLE:", row["short_description"])
    print("  DESC :", row["long_description"][:200].replace("\n", " "))
    print()

420 usable candidates
PLATFORM-31385 | PLATFORM | SWT | normal
  TITLE: Redraw problems: selecting text backwards
  DESC : eclipse 2.1I200302061700 linux-gtk j2sdk-1.4.1_01 2GHz dual-processor pentium  I'm seeing a variety of redraw bug.  The most noticeable and consistent is that if I make an extended text selection via 

JDT-18719 | JDT | UI | low
  TITLE: No code complete in JavaDoc for methods in the same type [javadoc]
  DESC : Build 20020602  1. Create the following class: public class A {   /**    * Doc for a    */   public void a() {   }   /**    * @see A#    */   public void b() {   } } 2. Put your cursor in the JavaDoc 

TIGERSTRIPE-320050 | TIGERSTRIPE | UI | normal
  TITLE: Edit button for stereotypes that can't be edited should be disabled
  DESC : Build Identifier:   See 'Steps to Reproduce'   Reproducible: Always  Steps to Reproduce: 1. Open an Association in editor 2. Under Stereotypes click on Add > Provisional > OK 3. Select stereotype > Ed

PDE-172675 | PDE | UI | no

## 6. Add hand-written test reports

The real data has no duplicate labels (every bug is "fixed"), so I wrote test cases myself. They use the `SYNTH-` prefix so they cannot be confused with real bugs.

- **3 reworded duplicates** (`SYNTH-DUP-1` to `3`): each rewords a bug that is in the index. `expected_original_id` is the answer key.
- **3 edge cases:** a vague report, a question that is not a bug, and an out-of-scope request. They have no severity or component, because those labels do not apply.

Together with the 15 sampled reports, this makes 21 test reports.

**Note:** The wording of the three edge cases was drafted with Claude and then reviewed by me.

In [7]:
test_df["case_type"] = "sampled"
test_df["expected_original_id"] = None

extra_cases = pd.DataFrame([
    # --- 3 reworded duplicates: copy the ORIGINAL's labels and id ---
    {"bug_id": "SYNTH-DUP-1", "product_name": "PLATFORM", "short_description": "Text selection drawn incorrectly when selecting backwards",
     "long_description": "On Eclipse 2.1 (I200302061700) using Linux-GTK with J2SDK 1.4.1_01 on a dual-processor Pentium 2GHz machine, multiple redraw issues occur. The most consistent problem appears when making an extended text selection in reverse (backwards), where the selection rendering becomes incorrect.", 
     "component_name": "SWT", "severity": "normal",
     "case_type": "duplicate", "expected_original_id": "PLATFORM-31385"},
    
    {"bug_id": "SYNTH-DUP-2", "product_name": "JDT", "short_description": "JavaDoc code completion missing for methods in the same class",
     "long_description": "In Build 20020602, JavaDoc code completion does not propose methods from the same type. For example, in a class containing methods a() and b(), placing the cursor inside the JavaDoc @see A# tag does not offer completion for method names defined in the same class.", 
     "component_name": "UI", "severity": "low",
     "case_type": "duplicate", "expected_original_id": "JDT-18719"},
     
    {"bug_id": "SYNTH-DUP-3", "product_name": "XTEND", "short_description": "Error: Primitive types cannot be used with assertJ in Eclipse",
     "long_description": "In a simple project using assertJ, calling assertThat(anyPrimitiveType) causes an exception when saving the file. Eclipse reports an error stating that constraints may not be primitives, preventing normal editing.", 
     "component_name": "Core", "severity": "normal",
     "case_type": "duplicate", "expected_original_id": "XTEND-485032"},

    # --- edge cases: labels are not applicable ---
    {"bug_id": "SYNTH-VAGUE-1", "product_name": None, "short_description": "Eclipse does not work",
     "long_description": "It stopped working after the last update. Please fix it as soon as possible.", "component_name": None, "severity": None,
     "case_type": "vague", "expected_original_id": None},
    {"bug_id": "SYNTH-QUESTION-1", "product_name": None, "short_description": "How do I change the font size in the editor?",
     "long_description": "I want the text in the Java editor to be bigger. Where can I change this setting?", "component_name": None, "severity": None,
     "case_type": "question", "expected_original_id": None},
    {"bug_id": "SYNTH-OOS-1", "product_name": None, "short_description": "Mobile banking app crashes when I log in",
     "long_description": "The app closes by itself every time I enter my password on my phone. I am using the latest version.", "component_name": None, "severity": None,
     "case_type": "out_of_scope", "expected_original_id": None},
])

EXPECTED_OUTCOME = {
    "sampled": "auto_triaged",
    "duplicate": "duplicate",
    "vague": "needs_human_review",
    "question": "rejected",
    "out_of_scope": "rejected",
}

test_df = pd.concat([test_df, extra_cases], ignore_index=True)
test_df["expected_outcome"] = test_df["case_type"].map(EXPECTED_OUTCOME)

print(test_df["case_type"].value_counts())
print(f"Total test reports: {len(test_df)}")

case_type
sampled         15
duplicate        3
vague            1
question         1
out_of_scope     1
Name: count, dtype: int64
Total test reports: 21


## 7. Check the test set

Each reworded duplicate must point to an original that exists in the index, and the labels must match the original. The final assertions also catch leftover placeholders, repeated IDs, and hidden characters.

In [8]:
dup_rows = extra_cases[extra_cases["case_type"] == "duplicate"]
for _, r in dup_rows.iterrows():
    orig = index_df[index_df["bug_id"] == r["expected_original_id"]]
    if orig.empty:
        print(f"{r['bug_id']}: original {r['expected_original_id']} NOT in index_df")
        continue
    o = orig.iloc[0]
    print(f"{r['bug_id']} vs {o['bug_id']}")
    print("  original title:", o["short_description"])
    print("  reworded title:", r["short_description"])
    for col in ["product_name", "component_name", "severity"]:
        print(f"  {col}: original={o[col]!r} reworded={r[col]!r}")

SYNTH-DUP-1 vs PLATFORM-31385
  original title: Redraw problems: selecting text backwards
  reworded title: Text selection drawn incorrectly when selecting backwards
  product_name: original='PLATFORM' reworded='PLATFORM'
  component_name: original='SWT' reworded='SWT'
  severity: original='normal' reworded='normal'
SYNTH-DUP-2 vs JDT-18719
  original title: No code complete in JavaDoc for methods in the same type [javadoc]
  reworded title: JavaDoc code completion missing for methods in the same class
  product_name: original='JDT' reworded='JDT'
  component_name: original='UI' reworded='UI'
  severity: original='low' reworded='low'
SYNTH-DUP-3 vs XTEND-485032
  original title: Eclipse Error Constraints may not be primitives with assertJ and primitive types
  reworded title: Error: Primitive types cannot be used with assertJ in Eclipse
  product_name: original='XTEND' reworded='XTEND'
  component_name: original='Core' reworded='Core'
  severity: original='normal' reworded='normal'


### How hard are the duplicates?

Word overlap (Jaccard) between each reworded title and its original: 0.33, 0.54 and 0.47. These are fairly easy duplicates, since many key words are shared. The result will show how well keyword search works on easy cases, not on hard ones.

In [9]:
def word_overlap(a: str, b: str) -> float:
    wa, wb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(wa & wb) / len(wa | wb)

for _, r in extra_cases[extra_cases["case_type"] == "duplicate"].iterrows():
    orig = index_df[index_df["bug_id"] == r["expected_original_id"]].iloc[0]
    print(r["bug_id"], round(word_overlap(r["short_description"], orig["short_description"]), 2))

SYNTH-DUP-1 0.33
SYNTH-DUP-2 0.54
SYNTH-DUP-3 0.47


## 8. Save

`agentic_system.ipynb` loads these files with `keep_default_na=False`. The empty labels of the edge cases become empty strings when reloaded.

In [10]:
Path("data").mkdir(exist_ok=True)
index_df.to_csv("data/index.csv", index=False)
test_df.to_csv("data/test.csv", index=False)
print(f"Saved index ({len(index_df)} rows) and test ({len(test_df)} rows)")

Saved index (500 rows) and test (21 rows)


In [11]:
dup_rows = test_df[test_df["case_type"] == "duplicate"]
assert dup_rows["expected_original_id"].isin(index_df["bug_id"]).all()
assert not test_df["short_description"].isin(["...", ""]).any()
assert test_df["bug_id"].is_unique
assert not test_df["bug_id"].str.contains("\u2011").any()
assert test_df["expected_outcome"].notna().all()
print("All checks passed")

All checks passed


## Known limitations of this data
- Only **fixed** bugs are included. The agent has never seen rejected or unresolved reports.
- Some reports are auto-generated and look alike, so keyword search may match on boilerplate.
- Labels cover 5 components and 3 severity levels, and come from one dataset.
- The 500-bug index is a small sample, so a real duplicate may not be in it.
- The duplicate and edge-case tests are written by me and are few (6 in total).